# LOB simulator on Colab (CUDA + OpenMP + MPI)

1. **Runtime > Change runtime type > T4 GPU**
2. Run the cells top to bottom (or **Runtime > Run all**).
3. The last cell downloads `results.zip` with all CSVs, plots and profiler reports.

Colab gives only 2 CPU threads, so the OpenMP scaling experiments are better run on a laptop or lab machine (see README). This notebook covers correctness, the GPU experiments and profiling.

In [ ]:
# Branch to test. Change to "main" once the pull request is merged.
BRANCH = "lob-simulator"
!nvidia-smi --query-gpu=name,compute_cap,driver_version,memory.total --format=csv

In [ ]:
import os
os.environ["OMPI_ALLOW_RUN_AS_ROOT"] = "1"          # Colab runs as root
os.environ["OMPI_ALLOW_RUN_AS_ROOT_CONFIRM"] = "1"
!apt-get -qq install -y openmpi-bin libopenmpi-dev > /dev/null && echo "MPI installed"
!rm -rf /content/lob && git clone -q -b {BRANCH} https://github.com/abhisaarguptaa/Stock-market-simulator-using-PCAP.git /content/lob
%cd /content/lob
!git log --oneline -1

## Build and unit tests

In [ ]:
!cmake -S . -B build -DLOB_USE_CUDA=ON -DLOB_USE_MPI=ON -DCMAKE_CUDA_ARCHITECTURES=75 > /dev/null && cmake --build build -j 2>&1 | tail -3
!./build/lob_tests | tail -8

## Correctness: every back end must produce the identical trade log

In [ ]:
!bash scripts/check_determinism.sh build

## Model realism: stylised facts of returns

In [ ]:
!mkdir -p results
!./build/lobsim --agents 64000 --instruments 16 --ticks 20000 --decide cuda --match omp --prices results/prices.csv --header
!python analysis/stylized_facts.py results/prices.csv --out results/facts
from IPython.display import Image, display
display(Image("results/facts/stylized_facts.png"))

## GPU experiments (CPU-vs-GPU crossover, growing agents, kernel ablations)
This is the longest step. Lower `TICKS` to make it faster.

In [ ]:
!TICKS=200 REPS=3 bash scripts/bench.sh gpu build

## Profiling with Nsight Systems and Nsight Compute
If Nsight Compute reports `ERR_NVGPUCTRPERM`, GPU counters are locked on this VM; the timeline (Nsight Systems) still works.

In [ ]:
!bash scripts/profile.sh build
!python analysis/roofline.py results/profile/ncu_sorted.csv results/profile/ncu_unsorted.csv --out results/plots/roofline.png

## Plots and summary

In [ ]:
!python analysis/scaling.py results
import glob
for p in sorted(glob.glob("results/plots/*.png")):
    print(p); display(Image(p))
print(open("results/summary.md").read())

In [ ]:
!zip -qr results.zip results
from google.colab import files
files.download("results.zip")